# Financial ML Model Training Notebook
Experiment with different models and features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
# Load training data
df = pd.read_csv('../data/training_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Feature Engineering

In [ ]:
# Create derived features
df['savings_rate'] = (df['monthly_income'] - df['last_month_spending']) / df['monthly_income']
df['spending_ratio'] = df['last_month_spending'] / df['balance']
df['goal_progress'] = df['balance'] / df['savings_goal']

print("New features created!")
df[['savings_rate', 'spending_ratio', 'goal_progress']].head()

## 4. Model Training

In [ ]:
# Prepare features and target
features = ['balance', 'monthly_income', 'age', 'savings_goal', 
            'last_month_spending', 'num_transactions', 'avg_transaction',
            'savings_rate', 'spending_ratio', 'goal_progress']

X = df[features]
y = df['next_month_spending']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.3f}")

## 5. Feature Importance

In [ ]:
# Plot feature importance
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.show()

## 6. Predictions vs Actual

In [ ]:
# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Spending')
plt.ylabel('Predicted Spending')
plt.title('Predictions vs Actual')
plt.show()

## 7. Test Single Prediction

In [ ]:
# Test with sample user
sample_user = {
    'balance': 45280,
    'monthly_income': 30000,
    'age': 28,
    'savings_goal': 10000,
    'last_month_spending': 15000,
    'num_transactions': 25,
    'avg_transaction': 600,
    'savings_rate': 0.5,
    'spending_ratio': 0.33,
    'goal_progress': 4.5
}

sample_df = pd.DataFrame([sample_user])
prediction = model.predict(sample_df[features])[0]

print(f"Predicted next month spending: ₹{prediction:.2f}")